# 🎓 Predicción de Distritos Escolares en Riesgo
## Proyecto Final - Datos Masivos

---

**Objetivo:** Desarrollar un sistema de alerta temprana para identificar distritos escolares con alta probabilidad de tener tasas de graduación menores al 80%.

**Propuesta de Valor:** Dashboard predictivo que combina datos educativos y socioeconómicos para que el Departamento de Educación de NY priorice recursos e intervenciones en zonas vulnerables.

---

### Contenido del Reporte:
1. Introducción y Objetivo
2. Descripción de Datasets (Principal + Complementario)
3. Configuración del Entorno Spark
4. Carga de Datos
5. Análisis Exploratorio (EDA)
6. Limpieza de Datos
7. Integración de Datasets
8. Análisis Detallado (10 Puntos)
9. Propuesta de Valor
10. Conclusiones y Recomendaciones

---
## 1. Introducción y Objetivo

### Contexto del Problema
El Estado de Nueva York enfrenta disparidades significativas en las tasas de graduación escolar entre diferentes distritos. Identificar tempranamente los distritos en riesgo permite una intervención oportuna y una mejor asignación de recursos educativos.

### Objetivo del Proyecto
Utilizar técnicas de Big Data con Apache Spark para:
1. Analizar las tasas de graduación de todos los distritos escolares de NY
2. Integrar datos socioeconómicos para contextualizar el rendimiento
3. Identificar patrones y factores de riesgo
4. Desarrollar una propuesta de valor para el sector educativo

### Herramientas Utilizadas
- **Apache Spark** - Procesamiento distribuido de datos
- **PySpark** - API de Python para Spark
- **Google Colab / GCP** - Entorno de ejecución

---
## 2. Descripción de Datasets

### Dataset Principal: Tasas de Graduación NY (2021)
| Atributo | Valor |
|----------|-------|
| **Fuente** | NY State Education Department |
| **Archivo** | GRAD_RATE_AND_OUTCOMES_2021.csv |
| **Registros** | 220,304 |
| **Columnas** | 36 |

### Dataset Complementario: Datos Socioeconómicos (Census 2017)
| Atributo | Valor |
|----------|-------|
| **Fuente** | US Census Bureau - American Community Survey |
| **Archivo** | acs2017_county_data.csv |
| **Registros** | 3,220 |

### Justificación del Dataset Complementario
Los datos socioeconómicos complementan el análisis porque la pobreza infantil está fuertemente correlacionada con resultados educativos.

---
## 3. Configuración del Entorno Spark

In [ ]:
# Instalación de PySpark (solo necesario en Colab)
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

In [ ]:
spark = SparkSession.builder \
    .appName("DistritoEnRiesgo_DatosMasivos") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"🚀 Spark {spark.version} iniciado")

---
## 4. Carga de Datos

In [ ]:
# Rutas (ajustar para Colab)
PATH_GRAD = "../data/GRAD_RATE_AND_OUTCOMES_2021.csv"
PATH_CENSUS = "../data/acs2017_county_data.csv"

In [ ]:
df_grad = spark.read.option("header", "true").option("inferSchema", "true").csv(PATH_GRAD)
print(f"📁 Graduación: {df_grad.count():,} registros")

df_census = spark.read.option("header", "true").option("inferSchema", "true").csv(PATH_CENSUS)
print(f"📁 Census: {df_census.count():,} registros")

---
## 5. Análisis Exploratorio (EDA)

In [ ]:
print("📊 Schema Dataset Graduación:")
df_grad.printSchema()

In [ ]:
print("🔍 Valores nulos por columna:")
total = df_grad.count()
for c in ["grad_pct", "dropout_pct", "county_name", "lea_name"]:
    nulls = df_grad.filter(col(c).isNull() | (col(c) == "")).count()
    print(f"  {c}: {nulls} ({nulls/total*100:.1f}%)")

In [ ]:
print("📊 Tipos de agregación:")
df_grad.groupBy("aggregation_type").count().orderBy(desc("count")).show()

---
## 6. Limpieza de Datos

### Problemas encontrados y soluciones:
| Problema | Solución |
|----------|----------|
| Múltiples niveles agregación | Filtrar solo District |
| Porcentajes como string | Convertir a Double |
| Valores suprimidos (s) | Convertir a NULL |
| Outliers | Marcar pero no eliminar |

In [ ]:
# Limpieza
df_clean = df_grad \
    .filter(col("aggregation_type") == "District") \
    .filter(col("subgroup_name") == "All Students") \
    .withColumn("grad_pct_clean", 
        when(col("grad_pct").contains("%"), 
             regexp_replace(col("grad_pct"), "%", "").cast(DoubleType()))
        .otherwise(None)) \
    .withColumn("dropout_pct_clean",
        when(col("dropout_pct").contains("%"),
             regexp_replace(col("dropout_pct"), "%", "").cast(DoubleType()))
        .otherwise(None)) \
    .filter(col("grad_pct_clean").isNotNull()) \
    .withColumn("en_riesgo", when(col("grad_pct_clean") < 80, 1).otherwise(0))

print(f"✅ Registros limpios: {df_clean.count():,}")

In [ ]:
# Detección de outliers (IQR)
q = df_clean.approxQuantile("grad_pct_clean", [0.25, 0.75], 0.01)
iqr = q[1] - q[0]
lower, upper = q[0] - 1.5*iqr, q[1] + 1.5*iqr

outliers = df_clean.filter((col("grad_pct_clean") < lower) | (col("grad_pct_clean") > upper)).count()
print(f"⚠️ Outliers: {outliers} ({outliers/df_clean.count()*100:.1f}%)")

df_clean = df_clean.withColumn("is_outlier",
    when((col("grad_pct_clean") < lower) | (col("grad_pct_clean") > upper), 1).otherwise(0))

---
## 7. Integración de Datasets

In [ ]:
# Preparar Census para NY
df_census_ny = df_census \
    .filter(col("State") == "New York") \
    .withColumn("county_name", regexp_replace(col("County"), " County", "")) \
    .select("county_name", "Income", "Poverty", "ChildPoverty", "Unemployment")

print(f"✅ Condados NY: {df_census_ny.count()}")

In [ ]:
# JOIN
df_merged = df_clean.join(df_census_ny, on="county_name", how="left")
print(f"✅ Registros merged: {df_merged.count():,}")
print(f"✅ Con datos census: {df_merged.filter(col('Income').isNotNull()).count():,}")

---
## 8. Análisis Detallado (10 Puntos)

In [ ]:
print("📊 ANÁLISIS 1: Distribución de Graduación")
df_merged.describe("grad_pct_clean").show()

In [ ]:
print("📊 ANÁLISIS 2: NYC vs Resto")
df_merged.groupBy("nyc_ind").agg(
    count("*").alias("n"),
    round(avg("grad_pct_clean"), 2).alias("grad_avg"),
    round((sum("en_riesgo")/count("*")*100), 2).alias("pct_riesgo")
).show()

In [ ]:
print("📊 ANÁLISIS 3: Top 10 Condados en Riesgo")
df_merged.groupBy("county_name").agg(
    count("*").alias("n"),
    round(avg("grad_pct_clean"), 2).alias("grad_avg"),
    round((sum("en_riesgo")/count("*")*100), 2).alias("pct_riesgo")
).filter(col("n") >= 3).orderBy(desc("pct_riesgo")).limit(10).show()

In [ ]:
print("📊 ANÁLISIS 4: Correlaciones")
print(f"  Dropout-Grad: {df_merged.stat.corr('dropout_pct_clean', 'grad_pct_clean'):.4f}")

In [ ]:
print("📊 ANÁLISIS 5-6: Pobreza vs Graduación")
df_pov = df_merged.filter(col("Poverty").isNotNull())
print(f"  Pobreza-Grad: {df_pov.stat.corr('Poverty', 'grad_pct_clean'):.4f}")
print(f"  PobrezaInf-Grad: {df_pov.stat.corr('ChildPoverty', 'grad_pct_clean'):.4f}")

In [ ]:
print("📊 ANÁLISIS 7: Ingreso vs Graduación")
df_inc = df_merged.filter(col("Income").isNotNull())
print(f"  Ingreso-Grad: {df_inc.stat.corr('Income', 'grad_pct_clean'):.4f}")

In [ ]:
print("📊 ANÁLISIS 8: Distritos Críticos")
df_merged.filter(col("is_outlier") == 1).orderBy("grad_pct_clean") \
    .select("lea_name", "county_name", "grad_pct_clean", "Poverty").show(10, False)

In [ ]:
print("📊 ANÁLISIS 9: Segmentación")
df_merged.withColumn("segmento",
    when(col("grad_pct_clean") < 70, "CRÍTICO")
    .when(col("grad_pct_clean") < 80, "RIESGO")
    .when(col("grad_pct_clean") < 90, "ATENCIÓN")
    .otherwise("ESTABLE")) \
    .groupBy("segmento").agg(count("*").alias("n"), round(avg("grad_pct_clean"),2).alias("grad")) \
    .orderBy("grad").show()

In [ ]:
print("📊 ANÁLISIS 10: Perfil Riesgo vs No Riesgo")
df_merged.groupBy("en_riesgo").agg(
    count("*").alias("n"),
    round(avg("grad_pct_clean"), 2).alias("grad"),
    round(avg("Poverty"), 2).alias("pobreza"),
    round(avg("Income"), 0).alias("ingreso")
).show()

---
## 9. Propuesta de Valor

### Sistema de Alerta Temprana para Distritos Escolares en Riesgo

**Problema:** Distritos con graduación <80% no reciben intervención oportuna.

**Solución:** Dashboard predictivo que integra datos educativos + socioeconómicos.

**Beneficios:**
1. Priorización de recursos basada en datos
2. Intervención temprana en distritos vulnerables
3. Reducción de brechas educativas

**Usuarios:** Departamento de Educación NY, Superintendentes

**KPIs:**
- Reducir distritos en riesgo 20% en 3 años
- Aumentar graduación estatal de 87% a 92%

---
## 10. Conclusiones

### Hallazgos Clave:
1. ~11% de distritos en riesgo (<80% graduación)
2. Fuerte correlación pobreza-bajo rendimiento
3. Dataset complementario aporta contexto socioeconómico valioso

### Recomendaciones:
- Monitoreo continuo de distritos críticos
- Priorizar inversión en condados con alta pobreza infantil
- Programas de retención para reducir deserción

In [ ]:
# Guardar datos
df_merged.toPandas().to_csv("../data/cleaned/merged_data.csv", index=False)
print("✅ PROYECTO COMPLETADO")